[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ach0000/DualScoreMatching/blob/main/tp/TP_log_probabilites.ipynb)

# Quelle est la probabilité d'une image ? Entropie, NLL, contenu et généralisation

Le réseau d'énergie donne $\log p_\theta(x)$ en une seule évaluation. On calcule la distribution des log-probabilités des images de validation d'ImageNet64 (figure 3), l'entropie et la NLL (tableau 2), on relie la probabilité au contenu des images (figure 4), on compare avec un modèle gaussien et l'on reprend l'expérience de généralisation (figure 2) à partir des énergies calculées par les auteurs.

**Matériel** : tourne sur un ordinateur personnel (CPU) en quelques minutes ; sur Colab, exécuter d'abord la cellule d'installation.

**Profils** : profil réduit (par défaut), 5 000 images (environ 3 minutes pour les log-probabilités, 6 minutes en tout sur CPU) et des grilles réduites ; profil GPU, les 50 000 images de validation, comme les auteurs. Notebooks des auteurs correspondants : `auteurs/energy_histogram_colab.ipynb`, `auteurs/cross_entropy_colab.ipynb`, `auteurs/affine_and_sparsity_colab.ipynb`, `auteurs/generalization_colab.ipynb`.

Le profil réduit est utilisé par défaut, même avec un GPU (le calcul se fait alors sur le GPU) ; pour les réglages des auteurs, remplacer `if False:` par `if True:` dans la cellule de réglages (profil GPU, non vérifié).

**Données** : la cellule « Données », après l'installation, télécharge les images (534 Mo, image-net.org) ; elle ne fait rien par défaut : remplacer `if False:` par `if True:` au premier lancement.

**Reprise** : les calculs longs sont mis en cache dans `resultats/cache/` ; une exécution interrompue reprend là où elle s'était arrêtée. Supprimer ce dossier pour tout recalculer.

In [ ]:
# Installation (Colab) : à exécuter en premier ; en local, ne fait que rendre le code importable
GITHUB_USER = "ach0000"  # propriétaire du fork
REPO = "DualScoreMatching"
BRANCH = "main"
# seuls ces fichiers du dépôt sont téléchargés (motifs de git sparse-checkout)
FICHIERS = ['/*.py',
            '/networks/',
            '/tp/tp_outils.py',
            '/models/finalclean_imagenet64_grayscale_lr0.0005_1Msteps10decays/',
            '/models/finalclean_imagenet64_color_lr0.0005_1Msteps10decays/',
            '/outputs/energies_generalization.pt',
            '/outputs/energies_test.pt']

import os, sys, subprocess
if "google.colab" in sys.modules:
    root = os.path.join("/content", REPO)
    if not os.path.isdir(os.path.join(root, ".git")):
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", "-b", BRANCH,
                        "https://github.com/%s/%s.git" % (GITHUB_USER, REPO), root], check=True)
    subprocess.run(["git", "-C", root, "sparse-checkout", "set", "--no-cone", *FICHIERS], check=True)
    subprocess.run(["git", "-C", root, "checkout", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'einops', 'PyWavelets'], check=True)
    os.chdir(os.path.join(root, "tp"))
else:
    racine = os.getcwd()
    while not os.path.exists(os.path.join(racine, "main.py")) and os.path.dirname(racine) != racine:
        racine = os.path.dirname(racine)
    os.chdir(os.path.join(racine, "tp"))
sys.path.insert(0, os.getcwd())
print("Répertoire courant :", os.getcwd())

In [ ]:
# Données (une fois par machine ; sur Colab, à chaque nouvelle session) : images de validation d'ImageNet64
# (50 000 images), téléchargées depuis image-net.org (fichier officiel, 534 Mo), lues et enregistrées dans
# data/ à la racine du dépôt. Rien n'est téléchargé par défaut : remplacer False par True ci-dessous.
if False:
    import tp_outils
    tp_outils.preparer_imagenet64_val()

## 0. Réglages

In [ ]:
%matplotlib inline
import math, os
import numpy as np
import torch
import matplotlib.pyplot as plt
import tp_outils as T
T.nommer_figures("TP_log_probabilites")

# Profil : réduit par défaut, que le calcul se fasse sur CPU ou sur GPU (T.DEVICE suit le matériel connecté).
# Réglages des auteurs (profil GPU, non vérifié, GPU indispensable) : remplacer False par True ci-dessous.
GPU = False
if False:
    GPU = torch.cuda.is_available()
N_IMAGES = 50_000 if GPU else 5_000        # images de validation (niveaux de gris)
N_COULEUR = 50_000 if GPU else 1_000       # images pour la NLL en couleur
N_GRILLE = 101 if GPU else 21              # grille des transformations affines
GRAINE = 0
T.fixer_graine(GRAINE)
print("calcul sur", T.DEVICE)
m = T.charger_modele("gris")
d = m.d

## 1. Log-probabilités des images de validation

Une seule évaluation du réseau par image, à $t = 0$ ; résultat en décibels par dimension,
$\frac{10}{d}\log_{10}p_\theta(x)$, avec $d = 4096$. Les images sont celles de la validation d'ImageNet64 des
auteurs, dans le même ordre : on compare nos valeurs à celles qu'ils ont calculées pour les 50 000 images
(`outputs/energies_test.pt`), ce qui permet, même sur CPU, de travailler ensuite sur le jeu complet.

In [ ]:
X = T.charger_imagenet64_val(N_IMAGES)
import time; debut = time.time()
logp = m.log_probabilites(X, taille_lot=500 if GPU else 250, cache="validation")
print("%d images en %.0f s" % (len(X), time.time() - debut))
e_aut, C_aut = T.energies_auteurs()
lp_aut = T.nats_vers_dB(e_aut, d)
ecart = logp - lp_aut[:len(logp)]
print("comparaison aux auteurs (%d images) : écart moyen %.3f, écart absolu moyen %.3f, max %.3f dB/dim"
      % (len(logp), ecart.mean(), ecart.abs().mean(), ecart.abs().max()))
print("(les auteurs appliquaient un retournement horizontal aléatoire aux images de test)")

print("log p (dB/dim) : moyenne %.2f, médiane %.2f, écart-type %.2f, min %.2f, max %.2f, étendue %.2f"
      % (logp.mean(), logp.median(), logp.std(), logp.min(), logp.max(), logp.max() - logp.min()))
h = -logp.mean().item()
bits_continus = h * math.log2(10) / 10
print("entropie différentielle estimée : %.2f dB/dim = %.3f bits/dim" % (h, bits_continus))
print("entropie discrète (cases de largeur 1/256) : %.2f bits/dim" % (8 + bits_continus))
print("nombre d'images 8 bits « typiques » : 10^%.0f sur 10^%.0f"
      % (d * (8 + bits_continus) * math.log10(2), d * 8 * math.log10(2)))
print("rapport des probabilités extrêmes : 10^%.0f" % (d * (logp.max() - logp.min()).item() / 10))
print("jeu complet (valeurs des auteurs, 50 000 images) : moyenne %.2f, médiane %.2f, écart-type %.2f, min %.2f, "
      "max %.2f, étendue %.2f dB/dim" % (lp_aut.mean(), lp_aut.median(), lp_aut.std(), lp_aut.min(), lp_aut.max(),
                                         lp_aut.max() - lp_aut.min()))

## 2. NLL en couleur (tableau 2)

Le tableau 2 de l'article donne la NLL du modèle couleur en bits par dimension, d'un modèle **discret** obtenu en
supposant la densité constante dans chaque case de quantification : $\text{NLL} = 8 + \frac{U_\theta(x)}{d\ln 2}$
($d = 3\times64\times64$). On recalcule aussi la constante de normalisation sur nos images.

In [ ]:
mc = T.charger_modele("couleur")
Xc = T.charger_imagenet64_val(N_COULEUR, gris=False)
e_c = mc.energie(Xc, 0.0, taille_lot=250)
print("NLL couleur (constante des auteurs) : %.3f bits/dim  (article : 3,36)" % T.nll_discrete_bits(e_c, mc.d).mean())
C_ici, _ = T.constante_normalisation(mc, Xc[:2000])
print("constante : auteurs %.1f, recalculée sur 2000 images %.1f nats -> NLL %.3f bits/dim"
      % (mc.constante, C_ici, T.nll_discrete_bits(e_c - mc.constante + C_ici, mc.d).mean()))
ce = mc.perf_test.cross_entropies   # entropies croisées des auteurs (énergie en nats), aux 17 niveaux de bruit
print("NLL enregistrée par les auteurs pendant l'entraînement (t = 1e-9) : %.3f bits/dim" % (8 + ce.to("bits")[-1]))

## 3. Histogramme et loi de Gumbel (figure 3)

Les auteurs ajustent une loi de Gumbel (loi limite du maximum de variables indépendantes) par maximum de
vraisemblance. On repère aussi la log-probabilité des images constantes (la meilleure) et du bruit uniforme sur
$[0,1]^d$.

In [ ]:
a = torch.linspace(0, 1, 101)
constantes = a[:, None, None, None] * torch.ones(1, 1, 64, 64)
lp_const = m.log_probabilites(constantes)
T.fixer_graine(GRAINE)
lp_bruit = m.log_probabilites(torch.rand(100, 1, 64, 64))
print("images constantes : max %.2f dB/dim pour l'intensité %.2f ; intensité 0,5 : %.2f ; image noire : %.2f"
      % (lp_const.max(), a[lp_const.argmax()], lp_const[50], lp_const[0]))
print("bruit uniforme : %.2f dB/dim en moyenne" % lp_bruit.mean())

position, echelle = T.ajuster_gumbel(logp)
print("Gumbel : position %.2f, échelle %.2f dB/dim (article : 9,57 et 3,17)" % (position, echelle))
print("Gumbel sur les 50 000 valeurs des auteurs : position %.2f, échelle %.2f" % T.ajuster_gumbel(lp_aut))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
grille = np.linspace(-5, 50, 500)
for ax, log in zip(axes, [False, True]):
    ax.hist(logp.numpy(), bins=50, density=True, alpha=0.7, log=log, label="données")
    ax.plot(grille, T.densite_gumbel(grille, position, echelle), "r", label="Gumbel")
    ax.axvline(lp_const.max(), color="tab:green", ls="--", label="image constante")
    ax.axvline(lp_bruit.mean(), color="tab:brown", ls="--", label="bruit uniforme")
    ax.set_xlabel("log p (dB/dim)"); ax.set_xlim(-5, 50)
    if log:
        ax.set_ylim(1e-5, 0.3)
axes[0].legend(fontsize=8); axes[0].set_ylabel("fréquence")
T.montrer("histogramme")
plt.figure(figsize=(4.5, 3))
plt.plot(a, lp_const); plt.xlabel("intensité de l'image constante"); plt.ylabel("log p (dB/dim)")
plt.grid(alpha=0.3)
T.montrer("images_constantes")

## 4. Images classées par probabilité

Les 20 images les moins probables et les 20 plus probables de nos images, puis les 15 images de référence des auteurs
(figure 3 de l'article), choisies sur le jeu complet pour que leurs log-probabilités soient régulièrement espacées.

In [ ]:
ordre = torch.argsort(logp)
T.afficher_images(X[ordre[:20]], ["%.1f" % v for v in logp[ordre[:20]]], ncol=10, titre="les moins probables",
                  nom="moins_probables")
T.afficher_images(X[ordre[-20:]], ["%.1f" % v for v in logp[ordre[-20:]]], ncol=10, titre="les plus probables",
                  nom="plus_probables")
X_ref = T.charger_imagenet64_val(indices=T.IDX_REF_AUTEURS)
lp_ref = m.log_probabilites(X_ref)
T.afficher_images(X_ref, ["%.1f" % v for v in lp_ref], ncol=15, taille=1.1,
                  titre="images de référence des auteurs", nom="references")

## 5. Luminosité et contraste (figure 4, gauche et milieu)

$\log p_\theta(a x + b)$ en fonction du contraste $a$ et de la luminosité $b$, pour une image de référence
(on ne garde que $a + b \le 1$, pour rester dans $[0,1]$), puis la coupe $b = 1/2$ pour les 15 images de
référence.

In [ ]:
A = torch.linspace(0, 1, N_GRILLE)
x_milieu = X_ref[7]
carte = torch.full((N_GRILLE, N_GRILLE), float("nan"))
for i, b in enumerate(A):
    lot = A[:, None, None, None] * x_milieu[None] + b
    garde = A + b <= 1 + 1e-6
    carte[i, garde] = m.log_probabilites(lot[garde])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
im = axes[0].imshow(carte.numpy(), origin="lower", extent=[0, 1, 0, 1], cmap="hot", aspect="auto")
plt.colorbar(im, ax=axes[0], label="log p(ax + b)")
axes[0].set_xlabel("a (contraste)"); axes[0].set_ylabel("b (luminosité)")
couleurs = plt.cm.plasma(np.linspace(0, 1, 15))
A2 = torch.linspace(0, 0.5, N_GRILLE)
coupes = []
for k in range(15):
    coupe = m.log_probabilites(A2[:, None, None, None] * X_ref[k][None] + 0.5)
    coupes.append(coupe)
    axes[1].plot(A2, coupe, color=couleurs[k])
axes[1].set_xlabel("a"); axes[1].set_ylabel("log p(ax + 1/2)"); axes[1].set_ylim(0, 50)
T.montrer("affine")
print("pente moyenne de log p(ax + 1/2) entre a = 0,1 et a = 0,5 : %.1f dB/dim par unité de contraste"
      % np.mean([((c[-1] - c[len(c) // 5]) / (A2[-1] - A2[len(A2) // 5])).item() for c in coupes]))

# homogénéité : énergie brute de a x (sans luminosité) comparée à a^2 U(x)
U1 = m.energie(X_ref, normaliser=False)
for fac in [0.5, 2.0]:
    Ua = m.energie(fac * X_ref, normaliser=False)
    print("a = %.1f : U(a x) / (a^2 U(x)) médian = %.3f" % (fac, (Ua / (fac ** 2 * U1)).median()))

## 6. Probabilité et parcimonie (figure 4, droite)

Rapport de participation des coefficients d'ondelettes de Haar, $\|c\|_1^2 / (n\|c\|_2^2) \in (0, 1]$ : petit pour
une image parcimonieuse (quelques grands coefficients).

In [ ]:
pr = T.rapport_participation(X)
pente, ordonnee = np.polyfit(pr.numpy(), logp.numpy(), 1)
r = np.corrcoef(pr.numpy(), logp.numpy())[0, 1]
plt.figure(figsize=(4.5, 3.6))
plt.hist2d(pr.numpy(), logp.numpy(), bins=80, range=((0, 0.6), (0, 30)), cmap="hot")
xx = np.array([0, 0.6]); plt.plot(xx, pente * xx + ordonnee, "w", alpha=0.6)
plt.xlabel("rapport de participation"); plt.ylabel("log p (dB/dim)")
T.montrer("parcimonie")
print("régression : log p = %.1f %+.1f x PR ; corrélation %.2f, part de variance expliquée R² = %.2f"
      % (ordonnee, pente, r, r ** 2))

## 7. Comparaison avec un modèle gaussien

Une gaussienne $\mathcal N(\mu, \Sigma)$ ajustée sur d'autres images de validation (moyenne et covariance
empiriques ; on ajoute à $\Sigma$ la variance du bruit de quantification, $1/(12\cdot 256^2)$, faute de quoi les
plus petites valeurs propres sont nulles). On compare les log-probabilités qu'elle donne aux images, et celles de ses
propres échantillons.

In [ ]:
Xg = T.charger_imagenet64_val(20_000, debut=N_IMAGES if N_IMAGES + 20_000 <= 50_000 else 0).flatten(1).double()
mu = Xg.mean(0)
Sigma = (Xg - mu).T @ (Xg - mu) / len(Xg) + torch.eye(d, dtype=torch.float64) / (12 * 256 ** 2)
L = torch.linalg.cholesky(Sigma)
logdet = 2 * torch.log(torch.diagonal(L)).sum()
def logp_gauss(Z):
    w = torch.linalg.solve_triangular(L, (Z.flatten(1).double() - mu).T, upper=False)
    e = 0.5 * (w ** 2).sum(0) + 0.5 * logdet + d / 2 * math.log(2 * math.pi)
    return T.nats_vers_dB(e, d).float()
lp_g = logp_gauss(X[:5000])
T.fixer_graine(GRAINE)
echantillons = (mu + (L @ torch.randn(d, 2000, dtype=torch.float64)).T).float()
lp_g_ech = logp_gauss(echantillons)
print("gaussienne, images réelles : moyenne %.2f dB/dim, écart-type %.2f" % (lp_g.mean(), lp_g.std()))
print("gaussienne, ses échantillons : moyenne %.2f dB/dim, écart-type %.3f (théorie sqrt(d/2) nats = %.3f dB/dim)"
      % (lp_g_ech.mean(), lp_g_ech.std(), T.nats_vers_dB(torch.tensor(math.sqrt(d / 2)), d).abs()))
print("réseau, images réelles : moyenne %.2f dB/dim, écart-type %.2f" % (logp[:5000].mean(), logp[:5000].std()))
print("corrélation log p réseau / log p gaussienne : %.2f" % np.corrcoef(logp[:5000], lp_g)[0, 1])
T.afficher_images(echantillons[:10].view(-1, 1, 64, 64).clamp(0, 1), titre="échantillons du modèle gaussien",
                  nom="echantillons_gaussiens")
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.hist(logp[:5000].numpy(), bins=50, alpha=0.6, density=True, label="réseau")
ax.hist(lp_g.numpy(), bins=50, alpha=0.6, density=True, label="gaussienne")
ax.set_xlabel("log p (dB/dim)"); ax.legend()
T.montrer("gaussienne")

## 8. Généralisation (figure 2)

Les auteurs ont entraîné deux réseaux d'énergie (couleur) sur $N$ images de deux sous-ensembles disjoints A et B
d'ImageNet64, pour $N = 1$ à $10^5$ ; les poids ne sont pas distribués, mais les énergies qu'ils donnent aux images
d'entraînement le sont (`outputs/energies_generalization.pt`) : `eA[N, "B"]` contient les énergies, par le
réseau B entraîné sur $N$ images, des 1 024 premières images de A (dont les $N$ premières sont les images
d'entraînement du réseau A de même taille).

In [ ]:
_, eA, eB = torch.load(os.path.join(T.RACINE, "outputs", "energies_generalization.pt"), map_location="cpu",
                       weights_only=False)
dc = 3 * 64 * 64
c = lambda e: T.nats_vers_dB(e, dc)
Ns = [10, 100, 1_000, 10_000, 100_000]
fig, axes = plt.subplots(1, len(Ns), figsize=(15, 3.2))
print("%8s %12s %14s %12s" % ("N", "écart RMS", "écart moyen |.|", "corrélation"))
for ax, N in zip(axes, Ns):
    n = min(N, 1000)
    a_A, b_A = c(eA[N, "A"][:n]), c(eA[N, "B"][:n])
    a_B, b_B = c(eB[N, "A"][:n]), c(eB[N, "B"][:n])
    ax.scatter(a_A, b_A, s=4, color="tab:green", label="images de A")
    ax.scatter(a_B, b_B, s=4, color="tab:red", label="images de B")
    ax.plot([0, 40], [0, 40], "k", lw=0.8); ax.set_xlim(0, 40); ax.set_ylim(0, 40); ax.set_aspect("equal")
    ax.set_title("N = %d" % N); ax.set_xlabel("log p_A(x)")
    ecart = torch.cat([a_A - b_A, a_B - b_B]); tous_a = torch.cat([a_A, a_B]); tous_b = torch.cat([b_A, b_B])
    print("%8d %9.3f dB %11.3f dB %12.4f" % (N, ecart.pow(2).mean().sqrt(), ecart.abs().mean(),
                                            np.corrcoef(tous_a, tous_b)[0, 1]))
axes[0].set_ylabel("log p_B(x)"); axes[0].legend(fontsize=7)
T.montrer("generalisation")
print("\nMémorisation : log p moyen (dB/dim) donné par le réseau A_N à ses images d'entraînement et aux autres images")
for N in [1, 10, 100]:
    print("N = %4d : images d'entraînement %7.2f ; autres images (non vues) %7.2f"
          % (N, c(eA[N, "A"][:N]).mean(), c(eA[N, "A"][N:]).mean()))